In [ ]:
import sys
from schwab.auth import easy_client, client_from_token_file
from schwab.client import Client
from schwab.streaming import StreamClient
import asyncio
import json
import threading
import datetime
import pandas as pd

In [ ]:
api_key = 'jJnw'
app_secret = 'vU'
callback_url = 'https://127.0.0.1'#:8182/'
token_path = './token.json'

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Assumes you've already created a token
client = client_from_token_file(token_path=token_path,api_key=api_key,pp_secret=app_secret)

In [ ]:
#client.get_permissions()

In [ ]:
token_age=client.token_age()
token_age_remaining = 7*24*3600-token_age
token_age_remaining_hr = token_age_remaining/3600
print("token_age_remaining:",token_age_remaining_hr//24, " Days ",token_age_remaining_hr%24, "Hour")

In [ ]:
#quote['AAPL']
account_info = client.get_account_numbers().json()
#account_id = account_info['accountId']
print(account_info)

In [ ]:
accountNumber=int(account_info[0]['accountNumber'])
accountHash = account_info[0]['hashValue']
print(accountNumber, accountHash)

In [ ]:
global counter
counter=0
candles_dict={}
symbol_list = ['SPY','QQQ','GLD','XLF','VXX','EEM','USO','TLT','XLE','VTI','VNQ','JNK','VGK','IWM','EWJ','IBB','SLV','EFA']

#def streamer(symbol_list,accountNumber):
stream_client = StreamClient(client, account_id=accountNumber)

def run_async(coroutine): 
    loop = asyncio.new_event_loop() 
    asyncio.set_event_loop(loop) 
    loop.run_until_complete(coroutine)

async def read_stream():
    await stream_client.login()
    global counter
    counter+=1
    def print_message(message):
        #print(type(message))
        #print(message)
        #adict  = json.dumps(message, indent=4)
        #print(adict)
        #print(type(adict))
        alist = message['content']
        #print(alist)
        timenow = datetime.datetime.now()
        for a in alist:
            symbol = a['key']
            try:
                price = a['LAST_PRICE']
            except:
                continue
            if symbol not in candles_dict.keys():
                candles_dict[symbol] = []
                candles = candles_dict[symbol]
                candle = {'datetime':timenow,'close':price}
                candles.append(candle)
            else:
                candles = candles_dict[symbol]
                candle = candles[-1]
                timemn = 60*timenow.hour+timenow.minute
                candlemn = 60*candle['datetime'].hour+candle['datetime'].minute
                if timemn>candlemn:
                    new_candle = {'datetime':timenow,'close':price}
                    candles.append(new_candle)
                else:
                    candle['close'] = price
            #candles_dict[symbol] = price
            #print(counter, symbol, price)

    # Always add handlers before subscribing because many streams start sending
    # data immediately after success, and messages with no handlers are dropped.
    #stream_client.add_nasdaq_book_handler(print_message)
    #await stream_client.nasdaq_book_subs(symbol_list)

    stream_client.add_level_one_equity_handler(print_message)
    await stream_client.level_one_equity_subs(symbol_list)#, fields='LAST_PRICE')

    while True:
        await stream_client.handle_message()

#asyncio.run(read_stream())
thread = threading.Thread(target=run_async, args=(read_stream(),)) 
thread.start() 
#thread.join()

In [ ]:
quote = client.get_quotes(["AAPL", "AMD"]).json()

In [ ]:
#aapl = quote['AAPL']

In [ ]:
quote['AAPL']['quote']['lastPrice']

In [ ]:
#streamer(symbol_list,accountNumber)

In [ ]:
candles_dict

In [ ]:
#candles_dict['JNK'][-1]['close']
def calc_signal(candles_dict):
    signal={}
    period=5
    for symbol, candles in candles_dict.items():
        if len(candles)<5:
            signal[symbol]=False
            continue
        
        sum=0
        for i in range(period):
            sum+=candles[-(i+1)]['close']
        avg = sum/period
        if candles[-1]['close'] > avg and candles[-2]['close'] < avg:
            signal[symbol] = True
        else:
            signal[symbol] = False
    return signal
    

In [ ]:
#df = pd.DataFrame(candles_dict)
#df
candles_dict

In [ ]:
signal=calc_signal(candles_dict)
signal

In [ ]:
sys.getsizeof(candles_dict)

In [ ]:
# Initialize an empty dictionary for DataFrame creation
def get_df():
    df_data = {}
    # Loop through the original dictionary and populate df_data
    for stock, entries in candles_dict.items():
        for entry in entries:
            datetime_val = entry['datetime'].strftime("%Y %M %D %H %M")
            close_val = entry['close']
            if datetime_val not in df_data:
                df_data[datetime_val] = {}
            df_data[datetime_val][stock] = close_val
    
    # Convert the dictionary to a DataFrame
    df = pd.DataFrame.from_dict(df_data, orient='index')
    display(df)
    # Display the DataFrame
    #print(df)
#get_df()

In [ ]:
from schwab.orders.equities import equity_buy_limit
from schwab.orders.common import Duration, Session
def place_order_using_signal():
    for k,v in signal.items():
        if v==True:
            price = str(0.05*(round(candles_dict[k][-1]['close'],2)//0.05))
            print(k,price)
            print(accountHash,type(accountNumber))
            order_id = client.place_order(accountHash, 
                                          equity_buy_limit(k, 1, price).set_duration(Duration.GOOD_TILL_CANCEL).set_session(Session.NORMAL).build())
            print(order_id)
            #break
#place_order_using_signal()

In [ ]:
def run_live(LIVE_RUNNER):
    

In [ ]:
thread = threading.Thread(target=run_live, args=(LIVE_RUNNER,)) 
thread.start() 